In [1]:
# Use the shell escape (!) to run the script on Colab's cloud filesystem
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


CUDA Available: True
GPU Device: NVIDIA L4


In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive to access your processed dataset
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define repository details (Using public HTTPS url)
repo_name = "SME_Credit_Risk"
repo_url = f"https://github.com/mirkosimunovic/{repo_name}.git"

# 3. Clone the public repo (or pull if it already exists)
if not os.path.exists(f"/content/{repo_name}"):
    print(f"\nCloning public repository: {repo_url}...")
    !git clone {repo_url}
else:
    print(f"\nRepository {repo_name} already exists. Pulling latest code changes...")
    %cd /content/{repo_name}
    !git pull
    %cd /content

# 4. Create the target directory inside the cloned repo
!mkdir -p /content/{repo_name}/data/processed

# 5. Copy the cleaned SME dataset from Google Drive
# NOTE: If you saved the CSV inside a specific folder in Google Drive, 
# adjust the source path below (e.g., "/content/drive/MyDrive/YourFolder/processed_sme_final.csv")
drive_source_path = "/content/drive/MyDrive/xAI_Banking_Paper/data/processed/processed_sme_final.csv"
colab_target_path = f"/content/{repo_name}/data/processed/processed_sme_final.csv"

if os.path.exists(drive_source_path):
    !cp "{drive_source_path}" "{colab_target_path}"
    print("\n✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project")
else:
    print(f"\n⚠️ WARNING: Could not find your dataset at: {drive_source_path}")
    print("Please check your file path inside your Google Drive side panel and update 'drive_source_path'.")

# 6. Change active directory to your repository root
%cd /content/{repo_name}
print(f"\nActive directory set to: {os.getcwd()}")

Mounted at /content/drive

Cloning public repository: https://github.com/mirkosimunovic/SME_Credit_Risk.git...
Cloning into 'SME_Credit_Risk'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 28 (delta 9), reused 24 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 106.84 KiB | 569.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.

✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project
/content/SME_Credit_Risk

Active directory set to: /content/SME_Credit_Risk


In [3]:
!git pull

# 1. Install heavy, CUDA-dependent system libraries first
!pip install "fknni[rapids12]" --extra-index-url=https://pypi.nvidia.com
!pip install faiss-gpu-cu12

# 2. Silently install the rest of our standard project requirements
!pip install -q -r requirements.txt


Already up to date.
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 147.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 175.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 129.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 86.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 GB 61.4 MB/s eta 0:00:00:00:0100:01
INFO: pip is looking at multiple versions of cugraph-cu12 to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 261.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 246.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 137.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 76.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!python scripts/trainer.py

Loaded data/processed/processed_sme_final.csv
  Shape: 852,308 rows x 15 features
  Target MIS_Status — Paid in Full (0): 704,679 | Default (1): 147,629
  Default rate: 0.1732

Starting stratified 5-fold CV (scaler fit on training fold only).
Imbalance handling: cost-sensitive weights — SMOTE is not used.

Fold 1/5  scale_pos_weight=4.7733
  Fold 1 | Logistic Regression  Recall=0.8368  AUC-ROC=0.8406  F1=0.5388
  Fold 1 | Random Forest        Recall=0.7595  AUC-ROC=0.9643  F1=0.8022
  Fold 1 | XGBoost              Recall=0.9304  AUC-ROC=0.9754  F1=0.8121
  Fold 1 | LightGBM             Recall=0.9283  AUC-ROC=0.9731  F1=0.7988
  Fold 1 | CatBoost             Recall=0.9292  AUC-ROC=0.9738  F1=0.8037

Fold 2/5  scale_pos_weight=4.7733
  Fold 2 | Logistic Regression  Recall=0.8361  AUC-ROC=0.8392  F1=0.5378
  Fold 2 | Random Forest        Recall=0.7625  AUC-ROC=0.9650  F1=0.8018
  Fold 2 | XGBoost              Recall=0.9295  AUC-ROC=0.9756  F1=0.8082
  Fold 2 | LightGBM             Recall=